# c41 final — entraînement unique + publication GitHub

Entraîne **uniquement** la recette gagnante de la phase 2 :

```text
c41-lower-alph500-b3 : BPE + NFC + lowercase + WhitespaceSplit
+ alphabet 500 + byte fallback + ha/sw/yo/am x3
```

Résultat attendu sur validation : **score 1.7440** (±0.005), guardrail **PASS**,
UNK = 0, vocab 10 000. La dernière cellule **publie** `models/optimized_c41-*/` et
`reports/c41_final.*` vers GitHub (`submissions/` n'est jamais touché ici —
la promotion vers le dossier de soumission se fait séparément).

**Ne rien modifier** : `Runtime → Run all`, puis coller le token GitHub (portée `repo`)
dans le champ masqué de la dernière cellule.


## 1. Installation (versions officielles)

`tokenizers==0.22.1` est **imposé** par le challenge (le checker officiel vérifie
l'égalité exacte de version).


In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy sentencepiece
import tokenizers
print("tokenizers:", tokenizers.__version__, "(attendu 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Installer tokenizers==0.22.1 (exigence officielle)"

## 2. Constantes, métrique officielle et guardrail

Copie exacte de la métrique officielle (`competition/metrics.py`) et des constantes
(`competition/constants.py`) du dépôt du challenge.


In [ ]:
# =============================================================================
# 2. Constantes + métrique OFFICIELLES (compétition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Constantes officielles (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Dataset officiel ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Paramètres du balayage (modifiables) ----------------------------------
VOCAB_SIZE = 10_000            # imposé par le challenge
MAX_TRAIN_DOCS = None          # None = tout le train (240 000) ; ex. 60_000 pour un pré-balayage rapide
BASELINE_REFERENCE_SCORE = 2.059977   # score officiel obtenu par 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- MÉTRIQUE OFFICIELLE ---------------------------------------------------
def count_words(text: str) -> int:
    """Mots = séparés par des espaces blancs (comme l'évaluateur officiel)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id émis pour un texte non représentable (comme l'évaluateur officiel)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = liste de (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Score officiel = moyenne des langues notées (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"langues notées manquantes : {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x moyenne(fertility brute des langues notées)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


# ---- MÉTRIQUE QUALITÉ (robustesse aux variantes, Kamali 2026) ----------------
# Le score officiel ne voit pas OÙ tombent les découpes : à score proche (±0.01),
# on sélectionne le tokenizer dont les tokens survivent aux variantes qui
# préservent le sens (casse, NFD, espaces multiples, ponctuation détachée).
QUALITY_SEED = 7
QUALITY_SAMPLES_PER_LANG = 150
QUALITY_TIEBAND = 0.01           # bande autour du meilleur score brut
QUALITY_VARIANTS = ("lower", "nfd", "dblspace", "detachpunct")

# Ponctuation détachée par la variante "detachpunct" (ASCII + guillemets + éthiopienne)
_PUNCT_DETACH = (".,;:!?()[]{}" + "'" + '"'
                 + "\u00ab\u00bb\u2019\u2026\u2013\u2014"
                 + "\u1361\u1362\u1363\u1364\u1365\u1366\u1367\u1368")


def quality_samples(val_rows, per_lang=QUALITY_SAMPLES_PER_LANG, seed=QUALITY_SEED):
    """Échantillon déterministe (graine fixe) pour la métrique qualité."""
    import random
    rng = random.Random(seed)
    by_lang = defaultdict(list)
    for lang, text in val_rows:
        by_lang[lang].append(text)
    samples = []
    for lang in LANGUAGES:
        pool = by_lang[lang]
        for i in rng.sample(range(len(pool)), min(per_lang, len(pool))):
            samples.append((lang, pool[i]))
    return samples


def _detach_punct(text):
    for p in _PUNCT_DETACH:
        if p in text:
            text = text.replace(p, f" {p} ")
    return text


def quality_variants(text):
    """4 variantes préservant le sens (le texte brut sert de référence)."""
    return {
        "lower": text.lower(),
        "nfd": unicodedata.normalize("NFD", text),
        "dblspace": re.sub(r"\s", "  ", text),
        "detachpunct": _detach_punct(text),
    }


def _jaccard(ids_a, ids_b):
    set_a, set_b = set(ids_a), set(ids_b)
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def measure_quality(tokenizer, samples):
    """Jaccard moyen (tokens partagés base/variante) par variante + moyenne."""
    base_ids = [e.ids for e in tokenizer.encode_batch(
        [t for _, t in samples], add_special_tokens=False)]
    per_variant = {}
    for variant in QUALITY_VARIANTS:
        var_ids = [e.ids for e in tokenizer.encode_batch(
            [quality_variants(t)[variant] for _, t in samples], add_special_tokens=False)]
        per_variant[variant] = (sum(_jaccard(a, b) for a, b in zip(base_ids, var_ids))
                                / len(samples))
    per_variant["mean"] = sum(per_variant.values()) / len(per_variant)
    return per_variant


def validate_tokenizer_file(path):
    """Contrôles officiels de validation (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("fichier > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulaire {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("une langue ne produit aucun token")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("un décodage est vide")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Métrique officielle chargée. Vocab max:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

## 3. Données officielles (entraînement sur `train` uniquement)


In [ ]:
# =============================================================================
# 3. Chargement du dataset officiel + préparation des textes
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nEntraînement :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation   :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")

quality_rows = quality_samples(val_rows)
print("Échantillon qualité :", f"{len(quality_rows):,}", "textes (150/langue × 6, graine fixe) —",
      len(QUALITY_VARIANTS), "variantes par texte")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "train inattendu"
assert len(val_rows) == 24_000, "validation inattendue"

## 4. Entraînement c41 (recette figée — ne rien modifier)


In [ ]:
# =============================================================================
# Recette c41-lower-alph500-b3 — FIGÉE, ne rien modifier
#   BPE + NFC + lowercase + WhitespaceSplit + alphabet 500 + byte fallback
#   + langues notées x3. Score attendu sur validation : 1.7440 (±0.005).
# =============================================================================
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFC, Lowercase, Sequence as NormSequence
from tokenizers.pre_tokenizers import WhitespaceSplit
from tokenizers.decoders import ByteFallback
from tokenizers.trainers import BpeTrainer

BYTE_TOKENS = [f"<0x{i:02X}>" for i in range(256)]
BOOST = {"ha": 3, "sw": 3, "yo": 3, "am": 3}
EXPECTED_SCORE = 1.7440


def corpus_iterator(train_by_lang, boost):
    """Textes du train en round-robin équilibré, langues notées x3."""
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if i % 100_000 == 0:
                    print(f"    ... {i:,} textes fournis")


print("Entraînement c41 sur le train officiel uniquement (240 000 textes)…")
tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=True))
tokenizer.normalizer = NormSequence([NFC(), Lowercase()])
tokenizer.pre_tokenizer = WhitespaceSplit()
trainer = BpeTrainer(vocab_size=10_000, min_frequency=5,
                     special_tokens=["[UNK]"] + BYTE_TOKENS,
                     limit_alphabet=500)
t0 = time.time()
tokenizer.train_from_iterator(corpus_iterator(train_by_lang, BOOST), trainer=trainer)
print(f"Entraînement terminé en {time.time() - t0:.1f} s")

# Tokens byte ordinaires (forme Llama-2) : retirés de added_tokens, le flag
# byte_fallback est (re)forcé — byte fallback intact, zéro [UNK].
payload = json.loads(tokenizer.to_str())
payload["added_tokens"] = [t for t in payload["added_tokens"]
                            if not (len(t["content"]) == 6 and t["content"].startswith("<0x"))]
payload["model"]["byte_fallback"] = True
tokenizer = Tokenizer.from_str(json.dumps(payload))
tokenizer.decoder = ByteFallback()
print("Vocabulaire :", tokenizer.get_vocab_size(with_added_tokens=True), "(attendu 10000)")


## 5. Évaluation officielle sur `validation`


In [ ]:
# =============================================================================
# Évaluation : métrique officielle sur validation + guardrail EN/FR
# =============================================================================
fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tokenizer)
score = competition_score(fertility, unk_rate)
raw, budget, breaches = guardrail(fertility)
penalised = penalised_scores(fertility, unk_rate)

print(f"{'langue':8s} {'fertility':>9s} {'unk_rate':>10s} {'score':>8s}")
for l in LANGUAGES:
    print(f"{l:8s} {fertility[l]:>9.4f} {unk_rate[l]:>10.6f} {penalised[l]:>8.4f}")
print(f"\nScore (moyenne ha/sw/yo/am) : {score:.4f} "
      f"(attendu {EXPECTED_SCORE:.4f}, écart {score - EXPECTED_SCORE:+.4f})")
print(f"Guardrail : budget {budget:.4f} | en {fertility['en']:.4f} | fr {fertility['fr']:.4f} "
      f"→ {'PASS' if not breaches else 'FAIL ' + str(breaches)}")
print(f"UNK total : {unk_total} | lignes lossy : {lossy:,} (blancs/casse jetés : normal)")
if abs(score - EXPECTED_SCORE) > 0.005:
    print("\n⚠️ ÉCART ANORMAL (> 0.005) — ne pas publier sans analyse.")
else:
    print("\n✓ Score conforme à la phase 2 (c41 = 1.7440).")


## 6. Sauvegarde du modèle + rapports


In [ ]:
# =============================================================================
# Sauvegarde : models/optimized_c41-lower-alph500-b3/ + reports/c41_final.*
# (submissions/ n'est JAMAIS touché ici — promotion manuelle séparée)
# =============================================================================
model_dir = MODEL_DIR / "optimized_c41-lower-alph500-b3"
model_dir.mkdir(parents=True, exist_ok=True)
tok_path = model_dir / "tokenizer.json"
tokenizer.save(str(tok_path))
print("Tokenizer sauvegardé :", tok_path, f"({tok_path.stat().st_size:,} octets)")

candidate_path = OUTPUT_ROOT / "tokenizer.json"
shutil.copy2(tok_path, candidate_path)

report = {
    "experiment": "c41_final",
    "recipe": {"name": "c41-lower-alph500-b3", "model": "bpe", "normalizer": "nfc_lower",
               "pre": "whitespace_split", "min_freq": 5, "limit_alphabet": 500,
               "byte_fallback": True, "boost": BOOST},
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_texts": sum(len(v) for v in train_by_lang.values()),
                "validation_rows": len(val_rows)},
    "score": score,
    "expected_score": EXPECTED_SCORE,
    "fertility": fertility,
    "unk_rate": unk_rate,
    "penalised": penalised,
    "unk_total": unk_total,
    "lossy_rows": lossy,
    "guardrail": {"budget": budget, "breaches": breaches,
                  "pass": not breaches},
    "vocab_size": tokenizer.get_vocab_size(with_added_tokens=True),
    "environment": {"tokenizers": __import__("tokenizers").__version__,
                    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
}
(REPORT_DIR / "c41_final.json").write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                           encoding="utf-8")
md = ["# c41 — entraînement final (recette phase 2)", "",
      f"- Score : **{score:.4f}** (attendu {EXPECTED_SCORE:.4f})",
      f"- Guardrail : {'PASS' if not breaches else 'FAIL ' + str(breaches)} "
      f"(budget {budget:.4f}, en {fertility['en']:.4f}, fr {fertility['fr']:.4f})",
      f"- UNK : {unk_total} | vocab : {tokenizer.get_vocab_size(with_added_tokens=True)}",
      f"- Modèle : `{tok_path}`", ""]
(REPORT_DIR / "c41_final.md").write_text("\n".join(md), encoding="utf-8")
print("Rapports :", REPORT_DIR / "c41_final.json", "|", REPORT_DIR / "c41_final.md")


## 7. Vérification avec le **checker officiel** du challenge

Télécharge `starter/utils.py` du dépôt officiel et exécute `profile_submission` sur
le tokenizer — **le même code** que celui utilisé pour valider les soumissions.


In [ ]:
# =============================================================================
# 9. Checker officiel (starter/utils.py du dépôt du challenge)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("utils.py officiel téléchargé")
    except Exception as exc:
        print("Téléchargement impossible :", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Checker officiel indisponible — utilisation des contrôles intégrés.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Contrôles intégrés complémentaires (équivalents à competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nContrôles officiels intégrés :", checks["checks"])
print("Erreurs :", checks["errors"] or "aucune")
print("Langues non lossless (round-trip) :", checks["lossy_languages"] or "aucune (lossless)")

## 8. Publier le tokenizer sur GitHub (script + token)

La cellule suivante **écrit le script** `push_artifacts_to_github.py`, et la dernière
l'**exécute** : un **champ masqué** s'affiche pour coller le token GitHub (portée `repo`).

Sont publiés : `models/optimized_c41-lower-alph500-b3/` + `reports/c41_final.*`
sur la branche `arena/01a092c6-tokenizer` (`main` et `submissions/` restent intacts).


In [ ]:
%%writefile push_artifacts_to_github.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Publie les artefacts du challenge vers votre dépôt GitHub (méthode 2 : token).

Le token est fourni par saisie **masquée** (recommandé), par variable
d'environnement, ou par le secret Colab ``GITHUB_TOKEN``. Il n'est **jamais**
affiché, jamais écrit sur disque, jamais commité : toutes les sorties passent par
``redact()``.

Ce qui est publié (par défaut) :
    models/**      tokenizer(s) entraîné(s)
    reports/**     rapports JSON / Markdown
    submissions/** (avec --include-submissions) dossier de soumission

Exemples
--------
Colab — recommandé (dans une cellule Python, champ masqué actif) :
    import sys, runpy
    sys.argv = ["push_artifacts_to_github.py", "--source", "/content", "--include-submissions"]
    try:
        runpy.run_path("/content/push_artifacts_to_github.py", run_name="__main__")
    except SystemExit as exc:
        print("code de sortie :", exc.code)

Colab — avec !python : un sous-processus n'a ni champ masqué ni Secrets, il faut
fournir le token autrement (secret exporté dans l'environnement, ou --token-file) :
    !python scripts/push_artifacts_to_github.py --source /content

Colab, en incluant le dossier de soumission :
    !python scripts/push_artifacts_to_github.py --source /content --include-submissions

Supprimer au passage un dossier de soumission obsolète (ancien slug) :
    python scripts/push_artifacts_to_github.py --source /content --include-submissions \\
        --prune-submissions

Local :
    python scripts/push_artifacts_to_github.py --repo . --source .

Vérifier sans rien publier :
    python scripts/push_artifacts_to_github.py --source . --no-push

Créer explicitement une branche inexistante :
    python scripts/push_artifacts_to_github.py --source . --branch nouvelle-branche --create-branch

Publier sur une autre branche / un autre dépôt :
    python scripts/push_artifacts_to_github.py --source . --branch main \
        --repo-url https://github.com/<user>/<repo>.git
"""

from __future__ import annotations

import argparse
import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/maick-code/tokenizer.git"
DEFAULT_BRANCH = "arena/01a092c6-tokenizer"   # branche de travail (main reste intacte)
DEFAULT_MESSAGE = "Artifacts: tokenizer.json + reports (run Colab)"
ARTIFACT_DIRS = ("models", "reports")
EXCLUDE_DIR_NAMES = {"__pycache__", ".ipynb_checkpoints", ".git"}
EXCLUDE_SUFFIXES = (".pyc", ".pyo", ".zip", ".tmp", ".log")

EXIT_OK, EXIT_ERROR, EXIT_MISSING, EXIT_UNSAFE = 0, 1, 2, 3


# --------------------------------------------------------------------------- #
# Utilitaires
# --------------------------------------------------------------------------- #
def log(message: str = "") -> None:
    print(message, flush=True)


def die(message: str, code: int) -> "NoReturn":  # noqa: F821
    log(f"\nERREUR : {message}")
    raise SystemExit(code)


def redact(text: str, token: str | None) -> str:
    """Supprime toute trace du token d'une sortie."""
    if not text:
        return ""
    if token:
        text = text.replace(token, "***")
    return text


def clone_dir_default() -> Path:
    if os.path.isdir("/content"):          # Google Colab
        return Path("/content/tokenizer")
    return Path.cwd() / ".push_clone"


# --------------------------------------------------------------------------- #
# Token
# --------------------------------------------------------------------------- #
def token_from_colab_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get("GITHUB_TOKEN")
        return value.strip() if value else None
    except Exception:
        return None


_COLAB_MASKED_FIELD_JS = r"""
new Promise((resolve) => {
  const box = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;margin-top:6px;'
                    + 'border:1px solid #c8c8c8;border-radius:6px;display:inline-block';
  const label = document.createElement('span');
  label.textContent = 'Colle ton token GitHub puis valide : ';
  const input = document.createElement('input');
  input.type = 'password';
  input.style.cssText = 'font-size:14px;padding:3px 5px;width:330px';
  const button = document.createElement('button');
  button.textContent = 'Enregistrer';
  button.style.cssText = 'margin-left:8px;padding:3px 12px';
  const done = () => {
    input.disabled = true; button.disabled = true;
    const value = input.value; box.remove(); resolve(value);
  };
  button.addEventListener('click', done);
  input.addEventListener('keydown', (event) => { if (event.key === 'Enter') done(); });
  box.appendChild(label); box.appendChild(input); box.appendChild(button);
  document.body.appendChild(box);
  input.focus();
})
"""


def token_from_colab_masked_field() -> str | None:
    """Champ de saisie masqué natif Colab (nécessite d'exécuter le script EN PROCESSUS).

    Fonctionne quand le script est lancé dans une cellule Python (``runpy``), pas
    avec ``!python`` : un sous-processus n'a pas accès à l'interface du notebook.
    """
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        value = output.eval_js(_COLAB_MASKED_FIELD_JS)
    except Exception as exc:
        log(f"Champ masqué Colab indisponible ({type(exc).__name__}) : repli sur la saisie classique.")
        return None
    if isinstance(value, str) and value.strip():
        log("Token saisi dans le champ masqué Colab (non affiché).")
        return value.strip().strip('"').strip("'")
    return None


def read_token(args: argparse.Namespace) -> str | None:
    """Token par ordre de priorité : --token-file, env, secret Colab, champ masqué Colab, saisie."""
    if args.token_file:
        path = Path(args.token_file)
        if not path.is_file():
            die(f"fichier de token introuvable : {path}", EXIT_ERROR)
        token = path.read_text(encoding="utf-8").strip()
        if token:
            log("Token lu depuis le fichier indiqué (--token-file).")
            return token

    for var in ("GITHUB_TOKEN", "GH_TOKEN"):
        token = os.environ.get(var)
        if token:
            log(f"Token récupéré depuis la variable d'environnement {var}.")
            return token.strip()

    token = token_from_colab_secret()
    if token:
        log("Token récupéré depuis le secret Colab 'GITHUB_TOKEN'.")
        return token

    if args.no_input:
        return None

    token = token_from_colab_masked_field()
    if token:
        return token

    prompt = "Colle ton token GitHub puis Entrée : "
    try:
        token = getpass.getpass(prompt)          # saisie masquée
    except Exception:
        try:
            token = input(prompt)                # repli si getpass indisponible
        except Exception:
            return None
    token = (token or "").strip().strip('"').strip("'")
    if token:
        log(f"Token saisi ({len(token)} caractères, non affiché).")
    return token or None


def authed_url(url: str, token: str | None) -> str:
    """URL https porteuse du token, uniquement pour github.com."""
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@")
    return url


# --------------------------------------------------------------------------- #
# Git
# --------------------------------------------------------------------------- #
def git(repo: Path | str | None, *args: str) -> subprocess.CompletedProcess:
    command = ["git"]
    if repo is not None:
        command += ["-C", str(repo)]
    return subprocess.run(command + list(args), capture_output=True, text=True)


def git_or_die(repo: Path | str | None, token: str | None, *args: str,
               what: str = "commande git") -> subprocess.CompletedProcess:
    result = git(repo, *args)
    if result.returncode != 0:
        die(f"{what} a échoué :\n{redact(result.stderr or result.stdout, token).strip()}",
            EXIT_ERROR)
    return result


# --------------------------------------------------------------------------- #
# Artefacts
# --------------------------------------------------------------------------- #
def best_model_tokenizer(source: Path) -> Path | None:
    """Chemin du tokenizer de la meilleure configuration du dernier balayage."""
    import json

    sweep = source / "reports" / "optimization_sweep.json"
    if not sweep.is_file():
        return None
    try:
        name = json.loads(sweep.read_text(encoding="utf-8"))["best"]["name"]
    except Exception:
        return None
    candidate = source / "models" / f"optimized_{name}" / "tokenizer.json"
    return candidate if candidate.is_file() else None


def prune_obsolete_submissions(source: Path) -> list[str]:
    """Supprime les dossiers submissions/<slug>/ obsolètes (tokenizer != meilleur modèle).

    Cas typique : après avoir renommé le SLUG, l'ancien dossier reste sur le disque et serait
    publié avec le nouveau — or le checker officiel exige exactement un répertoire de
    soumission. Seuls des dossiers dont le tokenizer.json diffère du meilleur modèle sont
    supprimés, et seulement s'il en reste plusieurs : le dossier courant est toujours conservé.
    """
    import hashlib

    subs = source / "submissions"
    if not subs.is_dir():
        return []
    dirs = sorted(p for p in subs.iterdir() if p.is_dir() and (p / "tokenizer.json").is_file())
    if len(dirs) < 2:
        return []
    best = best_model_tokenizer(source)
    digest = (lambda p: hashlib.sha256(p.read_bytes()).hexdigest())
    if best is not None and any(digest(d / "tokenizer.json") == digest(best) for d in dirs):
        keep = [d for d in dirs if digest(d / "tokenizer.json") == digest(best)]
    else:
        keep = [max(dirs, key=lambda d: d.stat().st_mtime)]
    removed = []
    for d in dirs:
        if d not in keep:
            shutil.rmtree(d)
            removed.append(d.name)
    if removed:
        log(f"dossiers de soumission obsolètes supprimés : {', '.join(removed)}")
        log(f"dossier conservé : {keep[0].name}")
    return removed


def collect_artifacts(source: Path, include_submissions: bool) -> list[str]:
    """Chemins relatifs (posix) des fichiers à publier, triés."""
    roots = list(ARTIFACT_DIRS) + (["submissions"] if include_submissions else [])
    files: list[str] = []
    for root in roots:
        base = source / root
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file():
                continue
            parts = set(path.relative_to(source).parts)
            if parts & EXCLUDE_DIR_NAMES or path.name.startswith("."):
                continue
            if path.suffix.lower() in EXCLUDE_SUFFIXES:
                continue
            files.append(path.relative_to(source).as_posix())
    return files


def safety_checks(source: Path, files: list[str], force: bool) -> list[str]:
    """Contrôles avant publication. Retourne la liste des avertissements bloquants."""
    import json

    problems: list[str] = []

    baseline = source / "reports" / "baseline_bpe_10k.json"
    if baseline.is_file():
        try:
            status = json.loads(baseline.read_text(encoding="utf-8")).get("status")
            if status != "computed_on_official_dataset":
                problems.append(
                    f"reports/baseline_bpe_10k.json : status = {status!r} "
                    "(run non conforme au dataset officiel)")
        except Exception as exc:
            problems.append(f"reports/baseline_bpe_10k.json illisible : {exc}")

    sweep = source / "reports" / "optimization_sweep.json"
    if sweep.is_file():
        try:
            payload = json.loads(sweep.read_text(encoding="utf-8"))
            rows = (payload.get("dataset") or {}).get("validation_rows")
            if rows != 24_000:
                problems.append(
                    f"reports/optimization_sweep.json : validation_rows = {rows} "
                    "(attendu 24 000 : le balayage n'a pas tourné sur le vrai dataset)")
        except Exception as exc:
            problems.append(f"reports/optimization_sweep.json illisible : {exc}")

    if not any(f.startswith("models/") and f.endswith("tokenizer.json") for f in files):
        problems.append("aucun models/**/tokenizer.json trouvé dans les artefacts")

    # Une soumission = UN dossier. Un dossier obsolète laissé par une exécution antérieure
    # (ancien slug, par exemple après avoir renommé SLUG) rendrait la PR invalide : le
    # checker officiel exige exactement un répertoire `submissions/<slug>/`.
    subs = source / "submissions"
    if subs.is_dir():
        slugs = sorted(p.name for p in subs.iterdir()
                       if p.is_dir() and (p / "tokenizer.json").is_file())
        if len(slugs) > 1:
            problems.append(
                f"plusieurs dossiers de soumission dans submissions/ : {', '.join(slugs)} "
                "(un seul slug est autorisé par PR ; supprimez les dossiers obsolètes)")

    if problems and not force:
        log("\n" + "!" * 74)
        log("PUBLICATION REFUSÉE — les artefacts semblent ne pas venir d'un run réel :")
        for problem in problems:
            log(f"  - {problem}")
        log("Corrigez le run, ou relancez avec --force pour publier quand même.")
        log("!" * 74)
        raise SystemExit(EXIT_UNSAFE)

    if problems:
        log("\nAVERTISSEMENT (--force) :")
        for problem in problems:
            log(f"  - {problem}")
    return problems


# --------------------------------------------------------------------------- #
# Programme principal
# --------------------------------------------------------------------------- #
def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Publie models/ et reports/ vers votre dépôt GitHub (méthode token).",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--source", default="/content" if os.path.isdir("/content") else ".",
                        help="répertoire contenant models/ et reports/ (défaut : /content ou .)")
    parser.add_argument("--repo", default=None,
                        help="clone git existant du dépôt (sinon clonage automatique)")
    parser.add_argument("--repo-url", default=DEFAULT_REPO_URL, help="URL https du dépôt")
    parser.add_argument("--branch", default=DEFAULT_BRANCH, help="branche cible")
    parser.add_argument("--message", default=DEFAULT_MESSAGE, help="message de commit")
    parser.add_argument("--token-file", default=None,
                        help="lire le token depuis un fichier (évite la saisie)")
    parser.add_argument("--no-input", action="store_true",
                        help="ne jamais demander le token de façon interactive")
    parser.add_argument("--no-push", action="store_true",
                        help="copier et commiter sans pousser")
    parser.add_argument("--prune-submissions", action="store_true",
                        help="supprimer les dossiers de soumission obsolètes (ancien slug) "
                             "avant publication : un seul slug est autorisé par PR")
    parser.add_argument("--include-submissions", action="store_true",
                        help="publier aussi submissions/**")
    parser.add_argument("--zip", action="store_true",
                        help="créer en plus une archive de secours dans --source")
    parser.add_argument("--force", action="store_true",
                        help="publier malgré les avertissements de conformité")
    parser.add_argument("--create-branch", action="store_true",
                        help="autoriser la création de la branche si elle n'existe pas")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    source = Path(args.source).resolve()
    branch = args.branch
    repo_url = args.repo_url

    log("=" * 74)
    log("Publication des artefacts vers GitHub")
    log("=" * 74)
    log(f"Source      : {source}")
    log(f"Dépôt       : {repo_url}")
    log(f"Branche     : {branch}")
    log(f"Artefacts   : {', '.join(ARTIFACT_DIRS + (('submissions',) if args.include_submissions else ()))}")
    log()

    if args.prune_submissions:
        prune_obsolete_submissions(source)

    files = collect_artifacts(source, args.include_submissions)
    if not files:
        die(f"aucun artefact trouvé dans {source} (attendu : models/, reports/)", EXIT_MISSING)

    log(f"{len(files)} fichier(s) à publier :")
    total = 0
    for rel in files:
        size = (source / rel).stat().st_size
        total += size
        log(f"  {size:>12,} o  {rel}")
    log(f"  {'-' * 12}")
    log(f"  {total:>12,} o  total")

    safety_checks(source, files, args.force)

    if args.zip:
        archive = source / "artifacts_backup.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            for rel in files:
                handle.write(source / rel, rel)
        log(f"\nArchive de secours : {archive} ({archive.stat().st_size:,} o)")

    token = read_token(args)

    repo = Path(args.repo).resolve() if args.repo else clone_dir_default()

    if not (repo / ".git").exists():
        if not token:
            die("aucun token fourni et pas de clone local : impossible de cloner.", EXIT_ERROR)
        log(f"\nClone de {repo_url} (branche {branch}) dans {repo} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", branch, authed_url(repo_url, token), str(repo)],
            capture_output=True, text=True)
        if clone.returncode != 0:
            die("clonage impossible (token invalide, branche inexistante ou réseau) :\n"
                f"{redact(clone.stderr or clone.stdout, token).strip()}", EXIT_ERROR)
        log("clone : OK")
    else:
        log(f"\nClone existant réutilisé : {repo}")

    if token:
        check = git(repo, "ls-remote", "--heads", authed_url(repo_url, token), branch)
        if check.returncode != 0:
            die("authentification refusée : vérifiez la portée `repo` du token,"
                " sa date d'expiration et le nom de la branche.", EXIT_ERROR)
        log("authentification : OK")

    # La branche cible doit exister : sans ce contrôle, une faute de frappe
    # créerait silencieusement une nouvelle branche distante.
    exists = git(None, "ls-remote", "--heads",
                 authed_url(repo_url, token) if token else repo_url, branch)
    if exists.returncode == 0 and not exists.stdout.strip():
        if args.create_branch:
            log(f"branche '{branch}' absente du dépôt : elle sera créée (--create-branch).")
        else:
            die(f"la branche '{branch}' n'existe pas sur {repo_url}.\n"
                "Vérifiez le nom (--branch), ou utilisez --create-branch pour la créer.",
                EXIT_ERROR)
    elif exists.returncode != 0 and not token:
        log("(impossible de vérifier la branche sans token : le push tranchera.)")

    # --- resynchronisation ---------------------------------------------------
    # Un clone Colab réutilisé (ou un clone créé dans une session précédente) peut être
    # en retard sur la branche distante : le commit local ne serait alors pas un
    # fast-forward et le push serait refusé. On se replace d'abord sur la tête distante ;
    # les artefacts étant recopiés juste après, rien n'est perdu.
    fetch = git(repo, "fetch", authed_url(repo_url, token) if token else repo_url, branch)
    if fetch.returncode == 0:
        ancestor = git(repo, "merge-base", "--is-ancestor", "FETCH_HEAD", "HEAD")
        if ancestor.returncode == 0:
            log("clone à jour avec la branche distante.")
        else:
            local = git(repo, "rev-parse", "--short", "HEAD").stdout.strip()
            remote = git(repo, "rev-parse", "--short", "FETCH_HEAD").stdout.strip()
            log(f"clone en retard ({local}) sur la branche distante ({remote}) : "
                "resynchronisation sur la tête distante (les artefacts sont recopiés ensuite).")
            git_or_die(repo, token, "checkout", "-B", branch, "FETCH_HEAD",
                       what=f"git checkout -B {branch} {remote}")
    else:
        log("fetch impossible (réseau ?) : on tente le push tel quel.")

    for rel in files:
        destination = repo / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, destination)
    log(f"{len(files)} fichier(s) copié(s) dans le clone.")

    git(repo, "config", "user.name", "Artifact Publisher")
    git(repo, "config", "user.email", "publisher@users.noreply.github.com")
    for root in {Path(rel).parts[0] for rel in files}:
        git_or_die(repo, token, "add", root, what=f"git add {root}")

    commit = git(repo, "commit", "-m", args.message)
    if commit.returncode == 0:
        log("commit : OK")
    elif "nothing to commit" in (commit.stdout + commit.stderr):
        log("commit : rien de nouveau (artefacts identiques)")
    else:
        die(f"commit impossible :\n{redact(commit.stderr or commit.stdout, token).strip()}",
            EXIT_ERROR)

    if args.no_push:
        log("\n--no-push : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    if not token:
        log("\nAucun token : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    push = git(repo, "push", authed_url(repo_url, token), f"HEAD:{branch}")
    if push.returncode != 0:
        die(f"push refusé :\n{redact(push.stderr or push.stdout, token).strip()}", EXIT_ERROR)

    log("push : OK")
    log()
    log(f"Publié sur {repo_url} (branche {branch}).")
    if "github.com" in repo_url:
        slug = repo_url.rstrip("/").removesuffix(".git")
        log(f"Vérifiez : {slug}/tree/{branch}")
    return EXIT_OK


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# =============================================================================
# Publication du tokenizer c41 vers GitHub (champ masqué Colab actif)
# Publie models/optimized_c41-*/ + reports/c41_final.* — submissions/ intouché.
# =============================================================================
import runpy
import sys
from pathlib import Path

SOURCE = Path(globals().get("OUTPUT_ROOT") or Path.cwd())
if not (SOURCE / "reports").is_dir():
    for candidate in (Path.cwd(), Path("/content")):
        if (candidate / "reports").is_dir():
            SOURCE = candidate
            break

SCRIPT_PATH = None
for candidate in (Path.cwd() / "push_artifacts_to_github.py",
                  SOURCE / "push_artifacts_to_github.py",
                  Path("/content/push_artifacts_to_github.py")):
    if candidate.is_file():
        SCRIPT_PATH = candidate.resolve()
        break
assert SCRIPT_PATH is not None, "la cellule %%writefile ci-dessus doit être exécutée d'abord"

sys.argv = [
    "push_artifacts_to_github.py",
    "--source", str(SOURCE),
    "--branch", "arena/01a092c6-tokenizer",
]

print("Exécution :", SCRIPT_PATH)
print("Arguments :", " ".join(sys.argv[1:]))
print("Un champ masqué « Colle ton token GitHub puis valide » va s'afficher.")
print()
try:
    runpy.run_path(str(SCRIPT_PATH), run_name="__main__")
except SystemExit as exc:
    print()
    print("Code de sortie du script :", exc.code,
          "| 0 = OK, 1 = erreur, 2 = artefacts manquants, 3 = publication refusée")
